# OPT-2.7B Fine-Tuning on Market-Informed Labels

**Thesis:** When Do Language Models Learn to Trade?

This notebook fine-tunes OPT-2.7B on Google Colab (A100/V100 GPU).
All other models (BERT, FinBERT, RoBERTa, OPT-350M) are trained locally.

**Before running:**
1. Set runtime to GPU: Runtime → Change runtime type → A100 (or V100)
2. Upload `matched_news_returns.parquet` to your Google Drive
3. Update the `DATA_PATH` below to point to your file

In [ ]:
# ============================================================
# Setup
# ============================================================
!pip install -q transformers accelerate datasets

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH to where your data lives in Google Drive
DATA_PATH = '/content/drive/MyDrive/thesis/data/matched_news_returns.parquet'
SAVE_DIR = '/content/drive/MyDrive/thesis/models/opt-2.7b_best'
RESULTS_DIR = '/content/drive/MyDrive/thesis/results/replication'

In [ ]:
# ============================================================
# Configuration
# ============================================================
import os
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_NAME = 'opt-2.7b'
HF_MODEL_ID = 'facebook/opt-2.7b'

# Grid search: learning rate only (batch and epochs fixed for memory/cost)
GRID_LR = [1e-5, 2e-5, 5e-5]
BATCH_SIZE = 4           # fixed due to memory
GRAD_ACCUM_STEPS = 4     # effective batch size = 16
EPOCHS = 3               # fixed due to compute cost
MAX_LENGTH = 512
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
PATIENCE = 3

# Temporal split dates
TRAIN_END = '2019-12-31'
TEST_START = '2020-01-01'

In [ ]:
# ============================================================
# Load and split data
# ============================================================
import pandas as pd
import numpy as np

torch.manual_seed(SEED)
np.random.seed(SEED)

df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

# Temporal split (no random shuffling!)
train_full = df[df['date'] <= TRAIN_END].copy()
test_df = df[df['date'] >= TEST_START].copy()

# Validation: last 20% of training period (temporal)
val_cutoff = train_full['date'].quantile(0.8)
train_df = train_full[train_full['date'] <= val_cutoff].copy()
val_df = train_full[train_full['date'] > val_cutoff].copy()

print(f'Train: {len(train_df):,} articles ({train_df["date"].min().date()} to {train_df["date"].max().date()})')
print(f'Val:   {len(val_df):,} articles ({val_df["date"].min().date()} to {val_df["date"].max().date()})')
print(f'Test:  {len(test_df):,} articles ({test_df["date"].min().date()} to {test_df["date"].max().date()})')
print(f'Label balance (train): {train_df["label"].mean():.3f} positive')

In [ ]:
# ============================================================
# Dataset class
# ============================================================
from torch.utils.data import Dataset, DataLoader

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
# ============================================================
# Training and evaluation functions
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def train_one_epoch(model, dataloader, optimizer, scheduler, device, grad_accum=1):
    model.train()
    total_loss = 0
    scaler = torch.amp.GradScaler('cuda')
    optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum

    return total_loss / len(dataloader)


def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1': f1_score(all_labels, all_preds, average='macro'),
        'precision': precision_score(all_labels, all_preds, average='macro'),
        'recall': recall_score(all_labels, all_preds, average='macro'),
    }

In [ ]:
# ============================================================
# Grid search over learning rate
# ============================================================
device = torch.device('cuda')
all_results = []
best_val_loss = float('inf')
best_lr = None

for lr in GRID_LR:
    print(f'\n{"="*60}')
    print(f'Training OPT-2.7B | lr={lr}, bs={BATCH_SIZE}x{GRAD_ACCUM_STEPS}, epochs={EPOCHS}')
    print(f'{"="*60}')

    # Load model fresh for each run
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForSequenceClassification.from_pretrained(
        HF_MODEL_ID, num_labels=2
    ).to(device)
    model.config.pad_token_id = tokenizer.eos_token_id

    # Datasets
    train_dataset = SentimentDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer, MAX_LENGTH)
    val_dataset = SentimentDataset(val_df['text'].tolist(), val_df['label'].tolist(), tokenizer, MAX_LENGTH)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = (len(train_loader) * EPOCHS) // GRAD_ACCUM_STEPS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    # Train with early stopping
    run_best_loss = float('inf')
    patience_counter = 0

    for epoch in range(EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, GRAD_ACCUM_STEPS)
        val_metrics = evaluate(model, val_loader, device)

        print(f'  Epoch {epoch+1}/{EPOCHS} | Train loss: {train_loss:.4f} | '
              f'Val loss: {val_metrics["loss"]:.4f} | Val acc: {val_metrics["accuracy"]:.4f}')

        if val_metrics['loss'] < run_best_loss:
            run_best_loss = val_metrics['loss']
            run_best_metrics = val_metrics.copy()
            patience_counter = 0

            # Save if this is the global best
            if run_best_loss < best_val_loss:
                best_val_loss = run_best_loss
                best_lr = lr
                model.save_pretrained(SAVE_DIR)
                tokenizer.save_pretrained(SAVE_DIR)
                print(f'  >> New best model saved (lr={lr})')
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    run_best_metrics['lr'] = lr
    all_results.append(run_best_metrics)

    # Clean up
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

# Save grid search results
results_df = pd.DataFrame(all_results)
results_df.to_csv(f'{RESULTS_DIR}/opt-2.7b_grid_search.csv', index=False)
print(f'\nBest learning rate: {best_lr}')
print(results_df.to_string())

In [ ]:
# ============================================================
# Evaluate best model on test set
# ============================================================
print('Loading best checkpoint...')
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)

test_dataset = SentimentDataset(test_df['text'].tolist(), test_df['label'].tolist(), tokenizer, MAX_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_metrics = evaluate(model, test_loader, device)
print(f'\nTest results for OPT-2.7B (best lr={best_lr}):')
print(f'  Accuracy:  {test_metrics["accuracy"]:.4f}')
print(f'  F1:        {test_metrics["f1"]:.4f}')
print(f'  Precision: {test_metrics["precision"]:.4f}')
print(f'  Recall:    {test_metrics["recall"]:.4f}')

# Generate predictions for portfolio construction
model.eval()
all_probs = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())

test_df_out = test_df.copy()
test_df_out['opt-2.7b_score'] = all_probs
test_df_out.to_parquet(f'{RESULTS_DIR}/opt-2.7b_test_predictions.parquet', index=False)
print(f'\nPredictions saved to {RESULTS_DIR}/opt-2.7b_test_predictions.parquet')
print('Download this file and place it in your local results/replication/ folder.')